In [1]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt


Looking in links: /kaggle/input/datasets/nhhsag12/colpali-dependency
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colpali_engine-0.3.15-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 1))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colbert_ai-0.2.21-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/bitarray-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/git_python-1.0.3-py2.py3-none-any.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/transformers-5.4.0-py3-no

## Cell 1 — Imports & Config

In [2]:
import os
import gc
import glob
import json
import pickle
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG — adjust paths to match your environment
# ==============================================================================

# Directory containing the split page-embedding pickle files produced by encoding notebook
PAGE_PKL_DIR   = "/kaggle/input/datasets/nhhsag12/dataset-encoded-page-colsmol"       # <-- change this

# ColPali model path (local checkpoint — used for ALL methods)
# COLPALI_BASE and COLPALI_LORA are already defined above; only LORA is needed
# for ColPali.from_pretrained() since it contains the merged weights + processor.

COLSMOL_BASE   = "/kaggle/input/models/nhhsag12/colsmolvlm-instruct-500m-base/pytorch/default/1"
COLSMOL_LORA   = "/kaggle/input/models/nhhsag12/colsmol-500m/pytorch/default/4"

# Dataset files
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PAGES_PARQUET    = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_pages.parquet"

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# Method config
TOPK_RATIOS        = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9
N_LAST_LAYERS_LIST = [4, 8, 16, 32]   # for Ours/Ablation method
NORMALIZE_MODES    = ['pre', 'post']   # for Ours/Ablation method
ATTN_N_LAYERS_LIST = [1, 2, 4, 8]     # for Attention Score method
KMEANS_ITERS       = 10               # for Spherical KMeans method
N_RANDOM_SEEDS     = 1                # for Random Pruning method (seeds to average)

print("Config loaded.")
print(f"TOPK_RATIOS       : {TOPK_RATIOS}")
print(f"N_LAST_LAYERS_LIST: {N_LAST_LAYERS_LIST}")
print(f"NORMALIZE_MODES   : {NORMALIZE_MODES}")
print(f"ATTN_N_LAYERS_LIST: {ATTN_N_LAYERS_LIST}")
print(f"KMEANS_ITERS      : {KMEANS_ITERS}")
print(f"N_RANDOM_SEEDS    : {N_RANDOM_SEEDS}")

Config loaded.
TOPK_RATIOS       : [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_LAST_LAYERS_LIST: [4, 8, 16, 32]
NORMALIZE_MODES   : ['pre', 'post']
ATTN_N_LAYERS_LIST: [1, 2, 4, 8]
KMEANS_ITERS      : 10
N_RANDOM_SEEDS    : 1


## Cell 2 — Load Page Embeddings from Pickle Files

In [3]:
# ==============================================================================
# Load all page embedding pickle files and concatenate them.
#
# Encoding notebook output format:
#   pickle.dump((encoded_quote, quote_indices), f)
#   - encoded_quote  : list[np.ndarray]  shape (n_tokens, D)
#   - quote_indices  : list[int]  — row_index in MMDocIR_pages.parquet
#
# If the documents were split into multiple pkl files, we merge them here.
# ==============================================================================

pkl_files = sorted(glob.glob(os.path.join(PAGE_PKL_DIR, "*.pkl")))
print(f"Found {len(pkl_files)} page PKL file(s):")
for p in pkl_files:
    print(f"  {p}")

if not pkl_files:
    raise FileNotFoundError(f"No pkl files found in {PAGE_PKL_DIR}")

all_page_embeddings = []   # list[np.ndarray]  each shape (n_tokens, D)
all_page_indices    = []   # list[int]  parquet row index, parallel to embeddings

for pkl_path in pkl_files:
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, tuple) and len(data) == 2:
        embs, idxs = data
    elif isinstance(data, dict):
        embs = data.get('embeddings', data.get('encoded_quote', []))
        idxs = data.get('indices', data.get('quote_indices', list(range(len(embs)))))
    else:
        raise ValueError(f"Unknown pkl format in {pkl_path}")
    all_page_embeddings.extend(list(embs))
    all_page_indices.extend(list(idxs))
    print(f"  Loaded {len(embs)} page embeddings from {os.path.basename(pkl_path)}")

print(f"\nTotal page embeddings : {len(all_page_embeddings)}")
print(f"Embedding shape sample: {all_page_embeddings[0].shape}")

Found 1 page PKL file(s):
  /kaggle/input/datasets/nhhsag12/dataset-encoded-page-colsmol/encoded_page_ColSmol_64.pkl
  Loaded 20395 page embeddings from encoded_page_ColSmol_64.pkl

Total page embeddings : 20395
Embedding shape sample: (1131, 128)


## Cell 3 — Load ColSmol Model & Processor for Live Query Encoding

In [4]:
from torch import nn
from transformers import Idefics3Model, Idefics3PreTrainedModel
from colpali_engine.models import ColIdefics3Processor

class ColIdefics3(Idefics3PreTrainedModel):
    """
    Initializes the ColIdefics3 model.

    Args:
        config : The model configuration.
        mask_non_image_embeddings (Optional[bool]): Whether to ignore all tokens embeddings
        except those of the image at inference.
        Defaults to False --> Do not mask any embeddings during forward pass.
    """

    def __init__(self, config, mask_non_image_embeddings: bool = False):
        super(ColIdefics3, self).__init__(config=config)
        self.model: Idefics3Model = Idefics3Model(config)
        self.dim = 128
        self.linear = nn.Linear(self.model.config.text_config.hidden_size, self.dim)
        self.mask_non_image_embeddings = mask_non_image_embeddings
        self.main_input_name = "doc_input_ids"
        self.post_init()

    def forward(self, *args, **kwargs):
        """
        Forward pass through Llama and the linear layer for dimensionality reduction

        Args:
        - input_ids (torch.LongTensor): The input tokens tensor.
        - attention_mask (torch.LongTensor): The attention mask tensor.

        Returns:
        - torch.Tensor: Embeddings of shape (batch_size, num_tokens, dim)
        """
        outputs = self.model(*args, **kwargs)
        last_hidden_states = outputs[0]  # (batch_size, sequence_length, hidden_size)
        proj = self.linear(last_hidden_states)
        # normalize l2 norm
        proj = proj / proj.norm(dim=-1, keepdim=True)
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            # Pools only the image embeddings
            image_mask = (kwargs["input_ids"] == self.config.image_token_id).unsqueeze(-1)
            proj = proj * image_mask
        return proj

    def forward_with_attentions(self, *args, **kwargs):
        """
        Same as forward() but returns (proj, raw_outputs) so the caller
        can access raw_outputs.attentions for importance scoring.
        """
        kwargs.pop("output_attentions", None)
        raw_outputs = self.model(
            *args,
            output_attentions=True,
            **kwargs,
        )
        last_hidden_states = raw_outputs[0]
        proj = self.linear(last_hidden_states)
        proj = proj / proj.norm(dim=-1, keepdim=True)
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            image_mask = (kwargs["input_ids"] == self.config.image_token_id).unsqueeze(-1)
            proj = proj * image_mask
        return proj, raw_outputs

In [5]:
from peft import PeftModel

query_model = ColIdefics3.from_pretrained(
    COLSMOL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="eager" # or eager
)
query_model = PeftModel.from_pretrained(
    query_model,
    COLSMOL_LORA
).eval()
query_processor = ColIdefics3Processor.from_pretrained(COLSMOL_LORA)


Loading weights:   0%|          | 0/490 [00:00<?, ?it/s]

In [6]:
# ==============================================================================
# Cell 3 — FIX: Attention Capture via AttentionCollector + encode_query_live
# ==============================================================================

from types import SimpleNamespace
import torch

# --------------------------------------------------------------------------
# Layer finder — skips vision/encoder stacks, picks the largest text stack.
# Cached after first scan (same model for all queries).
# --------------------------------------------------------------------------
_cached_text_layers_page = None

def _find_text_layers_page(model, force_rescan=False):
    global _cached_text_layers_page
    if _cached_text_layers_page is not None and not force_rescan:
        return _cached_text_layers_page

    candidates = []
    for name, mod in model.named_modules():
        if not name.endswith('.layers'):
            continue
        if not hasattr(mod, '__len__') or len(mod) == 0:
            continue
        is_vision = 'vision' in name.lower() or 'encoder' in name.lower()
        has_mlp   = hasattr(mod[0], 'mlp') or hasattr(mod[0], 'feed_forward')
        candidates.append({
            'name': name, 'module': mod,
            'n_layers': len(mod), 'is_vision': is_vision, 'has_mlp': has_mlp,
        })

    text_c = [c for c in candidates if not c['is_vision'] and c['has_mlp']]
    best   = max(text_c or candidates, key=lambda c: c['n_layers'], default=None)

    if best is None:
        raise RuntimeError(
            "Cannot find transformer text layers. Run `print(model)` to inspect structure."
        )
    print(f"[AttentionCollector] Using layer stack: '{best['name']}' "
          f"({best['n_layers']} layers)")
    _cached_text_layers_page = best['module']
    return _cached_text_layers_page


# --------------------------------------------------------------------------
# AttentionCollector — registers forward hooks on self_attn of each layer.
# Captures out[1] (attn_weights) when output_attentions=True flows through.
# --------------------------------------------------------------------------
class AttentionCollectorPage:
    def __init__(self):
        self.attentions = []
        self.hooks      = []

    def _find_attn_module(self, layer):
        for attr in ['self_attn', 'attention', 'attn']:
            if hasattr(layer, attr):
                return getattr(layer, attr)
        return None

    def register_hooks(self, model, layer_indices):
        """layer_indices: list of ints, negative allowed (e.g. [-4,-3,-2,-1])."""
        self.clear()
        self.remove_hooks()

        layers  = _find_text_layers_page(model)
        n_total = len(layers)

        for idx in layer_indices:
            idx_abs = idx if idx >= 0 else n_total + idx
            if not (0 <= idx_abs < n_total):
                continue
            attn_mod = self._find_attn_module(layers[idx_abs])
            if attn_mod is None:
                continue

            def _hook(module, inp, out):
                if isinstance(out, tuple) and len(out) >= 2 and out[1] is not None:
                    self.attentions.append(out[1].detach())

            self.hooks.append(attn_mod.register_forward_hook(_hook))

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    def clear(self):
        self.attentions.clear()


# --------------------------------------------------------------------------
# forward_with_attentions_colpali  (FIXED v2)
#
# Key changes:
#   1. Primary path: unwrap PeftModel → call ColIdefics3.forward_with_attentions()
#      directly to get native HF ModelOutput with .attentions.
#   2. Fallback path: hook-based AttentionCollectorPage (kept for robustness).
#   3. Paired with fixed encode_query_live() that uses the original tokenisation:
#      "Query: {question}<pad>*10" via processor.tokenizer() directly.
# --------------------------------------------------------------------------
def _unwrap_to_colidefics3(model):
    """
    Walk through PeftModel / LoraModel wrappers to reach the underlying
    ColIdefics3 instance that has forward_with_attentions().
    PeftModel structure: PeftModel.base_model → LoraModel.model → ColIdefics3
    """
    from torch import nn

    if isinstance(model, ColIdefics3):
        return model

    seen = {id(model)}
    queue = [model]
    for _ in range(6):
        if not queue:
            break
        nxt_queue = []
        for cur in queue:
            for attr in ('base_model', 'model'):
                if not hasattr(cur, attr):
                    continue
                child = getattr(cur, attr)
                if isinstance(child, ColIdefics3):
                    return child
                cid = id(child)
                if cid not in seen and isinstance(child, nn.Module):
                    seen.add(cid)
                    nxt_queue.append(child)
        queue = nxt_queue
    return None


def forward_with_attentions_colpali(model, inputs):
    """
    Run ColIdefics3 forward pass and return (proj, raw_outputs) where
    raw_outputs.attentions contains per-layer attention weight tensors.

    Strategy:
      1. (Primary) Unwrap PeftModel to reach ColIdefics3 and call its
         forward_with_attentions() method directly.
      2. (Fallback) Hook-based capture via AttentionCollectorPage.
    """
    kwargs = dict(inputs)

    # ── Primary path: direct forward_with_attentions ──────────────────
    base = _unwrap_to_colidefics3(model)
    if base is not None and hasattr(base, 'forward_with_attentions'):
        with torch.no_grad():
            proj, raw_outputs = base.forward_with_attentions(**kwargs)
        return proj, raw_outputs

    # ── Fallback path: hook-based capture ─────────────────────────────
    print("⚠️  WARNING: Could not unwrap to ColIdefics3.forward_with_attentions(). "
          "Falling back to hook-based attention capture.")

    collector = AttentionCollectorPage()
    layers    = _find_text_layers_page(model)
    collector.register_hooks(model, list(range(len(layers))))

    try:
        with torch.no_grad():
            kwargs['output_attentions'] = True
            proj = model(**kwargs)
    finally:
        collector.remove_hooks()

    attns = tuple(collector.attentions)
    if len(attns) == 0:
        print("⚠️  WARNING: AttentionCollector captured 0 attention tensors. "
              "Importance will fall back to uniform content mask. "
              "Check that attn_implementation='eager' is set on model load.")

    raw_outputs = SimpleNamespace(attentions=attns if attns else None)
    collector.clear()
    return proj, raw_outputs


# --------------------------------------------------------------------------
# encode_query_live  (FIXED — uses original tokenisation)
# --------------------------------------------------------------------------
def encode_query_live(question: str, processor, model, device: str):
    """
    Tokenise `question` and run a forward pass with attention outputs.
    Returns (proj, raw_outputs, inputs, encode_ms).

    NOTE: Tokenisation must match the original pipeline exactly:
      - Prefix with "Query: "
      - Suffix with "<pad>" * 10  (literal text, NOT special tokens)
      - Call processor() directly (NOT process_queries())
    Using process_queries() produces a different token sequence
    (empty prefix + special-token suffix) which degrades SVD importance
    scoring because:
      1. Missing "Query: " prefix changes the model's internal
         representation — ColSMoL was fine-tuned with this prefix.
      2. Special-token suffixes get masked out by build_content_mask_qwen,
         whereas literal "<pad>" text tokens are treated as regular
         content and participate in attention patterns, giving the
         importance scorer more signal to work with.
    """
    q_text = f"Query: {question}" + "<pad>" * 10
    inputs = processor(
        text=[q_text], return_tensors="pt", padding="longest",
    ).to(device)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        proj, raw_outputs = forward_with_attentions_colpali(model, inputs)

    torch.cuda.synchronize()
    encode_ms = (time.perf_counter() - t0) * 1000.0

    return proj, raw_outputs, inputs, encode_ms

print("✅ Fixed attention capture functions loaded.")

✅ Fixed attention capture functions loaded.


## Cell 4 — Build Page → Document Mapping and QA Pairs

In [7]:
# DEBUG CELL — run this before the QA mapping cell
import json

print("Loading MMDocIR_pages.parquet...")
pages_df = pd.read_parquet(PAGES_PARQUET)
print(f"  Pages parquet: {len(pages_df)} rows, columns: {list(pages_df.columns)}")

# Build reverse lookup and per-doc lookup first (needed for debug checks)
parquet_row_to_embed_idx = {parquet_row: embed_idx
                            for embed_idx, parquet_row in enumerate(all_page_indices)}

embedded_rows = pages_df.iloc[all_page_indices].copy()
embedded_rows['embed_idx']     = list(range(len(all_page_indices)))
embedded_rows['join_doc_name'] = embedded_rows['doc_name'].str.replace('.pdf', '', regex=False)
embedded_rows['passage_id']    = embedded_rows['passage_id'].astype(str)

doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby('join_doc_name')}
avail_docs      = set(doc_page_lookup.keys())

# 1. What does passage_id look like in the parquet?
print("=== pages_df sample ===")
print(pages_df[['doc_name', 'passage_id']].head(10).to_string())
print(f"\npassage_id dtype: {pages_df['passage_id'].dtype}")
print(f"passage_id sample values: {pages_df['passage_id'].iloc[:5].tolist()}")

# 2. What does the annotation's page_id look like?
print("\n=== annotation sample ===")
with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        doc_data = json.loads(line.strip())
        doc_name = doc_data['doc_name'].replace('.pdf', '')
        if doc_name in avail_docs:
            print(f"doc_name : {doc_name}")
            print(f"domain   : {doc_data.get('domain')}")
            for qa in doc_data['questions'][:3]:
                pid = qa.get('page_id')
                print(f"  Q: {qa['Q'][:60]}")
                print(f"  page_id: {repr(pid)}  type={type(pid).__name__}")
            break

# 3. Check doc_name overlap
print("\n=== doc_name overlap check ===")
annot_docs = set()
with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line.strip())
        annot_docs.add(d['doc_name'].replace('.pdf', ''))
print(f"Annotation docs : {len(annot_docs)}")
print(f"Embedded docs   : {len(avail_docs)}")
print(f"Intersection    : {len(annot_docs & avail_docs)}")
print(f"Sample annot    : {sorted(annot_docs)[:3]}")
print(f"Sample embedded : {sorted(avail_docs)[:3]}")

Loading MMDocIR_pages.parquet...
  Pages parquet: 20395 rows, columns: ['doc_name', 'domain', 'passage_id', 'image_path', 'image_binary', 'ocr_text', 'vlm_text']
=== pages_df sample ===
       doc_name passage_id
0  2310.05634v2          0
1  2310.05634v2          1
2  2310.05634v2          2
3  2310.05634v2          3
4  2310.05634v2          4
5  2310.05634v2          5
6  2310.05634v2          6
7  2310.05634v2          7
8  2310.05634v2          8
9  2310.05634v2          9

passage_id dtype: object
passage_id sample values: ['0', '1', '2', '3', '4']

=== annotation sample ===
doc_name : PH_2016.06.08_Economy-Final
domain   : Research report / Introduction
  Q: According to the report, how do 5% of the Latinos see econom
  page_id: [4]  type=list
  Q: According to the report, which one is greater in population 
  page_id: [18, 19]  type=list
  Q: From this report, which subgroup among Hispanics has gained 
  page_id: [13]  type=list

=== doc_name overlap check ===
Annotation docs :

In [8]:
# ==============================================================================
# Cell 4 — Build Page → Document Mapping and QA Pairs
#
# FIX: Each qa_pair now stores the document's page embed index range
# (doc_embed_indices) and ground truth as LOCAL indices within that range.
# This matches the eval notebook which scores each query against only its
# own document's pages.
# ==============================================================================

print("\nBuilding QA pairs from annotations...")

qa_pairs = []
q_global = 0

with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            doc_data = json.loads(line.strip())
        except Exception:
            continue

        doc_name = doc_data['doc_name'].replace('.pdf', '')
        domain   = doc_data.get('domain', 'General')

        if doc_name not in avail_docs:
            q_global += len(doc_data.get('questions', []))
            continue

        doc_pages = doc_page_lookup[doc_name]

        # Get the global embed indices for ALL pages in this document
        doc_all_embed_idxs = sorted(doc_pages['embed_idx'].tolist())
        # Build a map from global embed idx → local idx within this document
        global_to_local = {g: l for l, g in enumerate(doc_all_embed_idxs)}

        for qa in doc_data.get('questions', []):
            question_text = qa['Q']
            gt_page_ids   = qa.get('page_id', [])

            if not isinstance(gt_page_ids, list):
                gt_page_ids = [gt_page_ids]
            gt_page_ids_str = [str(p) for p in gt_page_ids]

            gt_embed_idxs = []
            for pid_str in gt_page_ids_str:
                matched = doc_pages[doc_pages['passage_id'] == pid_str]
                gt_embed_idxs.extend(matched['embed_idx'].tolist())

                if not matched.shape[0]:
                    matched_full = embedded_rows[
                        (embedded_rows['passage_id']    == pid_str) &
                        (embedded_rows['join_doc_name'] == doc_name)
                    ]
                    gt_embed_idxs.extend(matched_full['embed_idx'].tolist())

            gt_embed_idxs = list(set(gt_embed_idxs))

            if gt_embed_idxs:
                # Convert ground truth to LOCAL indices within this document
                gt_local_idxs = [global_to_local[g] for g in gt_embed_idxs if g in global_to_local]

                qa_pairs.append({
                    'question':           question_text,
                    'gt_embed_indices':   gt_embed_idxs,        # global (kept for backward compat)
                    'gt_local_indices':   gt_local_idxs,         # LOCAL within document
                    'doc_embed_indices':  doc_all_embed_idxs,    # all page embed indices for this doc
                    'doc_name':           doc_name,
                    'domain':             domain,
                })

            q_global += 1

qa_pairs = [q for q in qa_pairs if q.get('gt_local_indices')]
print(f"Total QA pairs built : {len(qa_pairs)}")
print(f"QA pairs after filter: {len(qa_pairs)}")


Building QA pairs from annotations...
Total QA pairs built : 1658
QA pairs after filter: 1658


## Cell 5 — Core Utility Functions (MaxSim, Metrics, Latency Tracker)

In [9]:
# ==============================================================================
# Shared utilities: doc matrix builder, MaxSim, metrics, latency tracker
# ==============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def build_doc_matrix(embeddings, device):
    """
    Convert list of np.ndarray embeddings to a padded (n_docs, max_len, D) tensor.
    Returns (doc_matrix, doc_mask).
    """
    arrays   = [torch.from_numpy(e).float() for e in embeddings]
    max_len  = max(a.shape[0] for a in arrays)
    D        = arrays[0].shape[1]
    n        = len(arrays)
    mat      = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask     = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    q_norm   : (N_q, D)   — query tokens, L2-normalized
    doc_matrix: (n_docs, max_len, D)
    doc_mask : (n_docs, max_len)  bool
    Returns  : (N_q, n_docs)  per-token MaxSim scores
    """
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)   # (N_q, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                             # (N_q, n_docs)


# ---------- Metrics (matching notebook-eval-mmdocir-page.md) ----------
# FIX: Use set-based recall/precision/ndcg from the eval notebook,
# NOT binary first-hit metrics.

def precision(retrieved, ground_truth):
    true_positives = len(set(retrieved) & set(ground_truth))
    return true_positives / len(retrieved) if len(retrieved) > 0 else 0

def recall(retrieved, ground_truth):
    true_positives = len(set(retrieved) & set(ground_truth))
    return true_positives / len(ground_truth) if ground_truth else 0

def ndcg(retrieved, ground_truth):
    ideal_dcg = sum(1 / np.log2(i + 2) for i in range(min(len(retrieved), len(ground_truth))))
    dcg = sum(1 / np.log2(i + 2) for i in range(len(retrieved)) if retrieved[i] in ground_truth)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

def average_precision(retrieved, ground_truth):
    true_positives = 0
    avg_prec = 0.0
    for i, item in enumerate(retrieved):
        if item in ground_truth:
            true_positives += 1
            avg_prec += true_positives / (i + 1)
    return avg_prec / true_positives if true_positives else 0

def mean_reciprocal_rank(retrieved, ground_truth):
    for i, item in enumerate(retrieved):
        if item in ground_truth:
            return 1 / (i + 1)
    return 0

def top_k_indices(scores, k):
    """Return indices of top-k scores in descending order (matches eval notebook)."""
    indexed_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    if k <= len(scores):
        return [index for index, score in indexed_scores[:k]]
    else:
        return [index for index, score in indexed_scores]

def eval_metrics(retrieved, gt, topk):
    """Compute all metrics for a single query at a given topk, matching eval notebook."""
    return {
        'precision': precision(retrieved, gt),
        'recall':    recall(retrieved, gt),
        'ndcg':      ndcg(retrieved, gt),
        'map':       average_precision(retrieved, gt),
        'mrr':       mean_reciprocal_rank(retrieved, gt),
    }

def compute_ndcg(ranked, gt, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt):
    for r, i in enumerate(top_k):
        if i in gt: return r + 1
    return -1

def hit_metrics(top_indices, gt_local, topk_list=[1, 3, 5]):
    """
    Compute per-document set-based metrics matching the eval notebook.
    top_indices: ranked list of LOCAL page indices (from top_k_indices)
    gt_local:    list of ground-truth LOCAL page indices
    topk_list:   list of K values to evaluate
    Returns dict with r@K, ndcg@K, map@K, mrr for each K.
    """
    result = {}
    for k in topk_list:
        topk = top_indices[:k]
        result[f'r{k}']    = recall(topk, gt_local)
        result[f'n{k}']    = ndcg(topk, gt_local)
        result[f'map{k}']  = average_precision(topk, gt_local)
        result[f'mrr{k}']  = mean_reciprocal_rank(topk, gt_local)
    return result

TOPK_EVAL = [1, 3, 5, 10]  # K values to report, matching eval notebook

def _init_metric():
    m = {'count': 0}
    for k in TOPK_EVAL:
        m[f'r{k}'] = 0.0
        m[f'n{k}'] = 0.0
    return m

def _add_metric(dst, src):
    for k in TOPK_EVAL:
        dst[f'r{k}']  += float(src[f'r{k}'])
        dst[f'n{k}']  += float(src[f'n{k}'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    header = f"{'Method':<35}"
    for k in TOPK_EVAL:
        header += f" {'R@'+str(k):>7}"
    for k in TOPK_EVAL:
        header += f" {'nDCG@'+str(k):>9}"
    print(header)
    print("-" * (35 + 7*len(TOPK_EVAL) + 9*len(TOPK_EVAL)))
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        row = f"{key:<35}"
        for k in TOPK_EVAL:
            row += f" {m[f'r{k}']/cnt*100:6.2f}%"
        for k in TOPK_EVAL:
            row += f" {m[f'n{k}']/cnt:8.4f}"
        print(row)


# ---------- Efficiency Tracker (FLOPs-based — paper-grade metric) ----------

class EfficiencyTracker:
    """
    Tracks per-ratio efficiency of MaxSim scoring via theoretical FLOPs
    and actual query token counts.

    For each (query, ratio) observation, records:
      - N_q : number of query vectors fed to MaxSim
      - N_d : total document tokens scored against (sum of valid tokens)
      - D   : embedding dimension

    MaxSim FLOPs for one query against a document set:
      dot products  : 2 × N_q × Σ(L_d) × D       (einsum 'qd,nld->qnl')
      max reduction :     N_q × Σ(L_d)
      sum reduction :     N_q × n_pages
    Dominant term: 2 × N_q × total_doc_tokens × D

    Since D and the document side are fixed across methods at the same ratio,
    FLOPs scale linearly with N_q → the "FLOPs ratio" is simply
    avg(N_q_reduced) / avg(N_q_full).

    Usage:
        tracker = EfficiencyTracker("Hierarchical Ward Pool", D=128)
        tracker.add(ratio, n_q_tokens, n_doc_tokens, n_pages)
        tracker.report()
        df = tracker.to_dataframe()
    """
    def __init__(self, method_name: str, D: int = 128):
        self.name = method_name
        self.D    = D
        self.data = {}   # ratio -> list[dict]

    def add(self, ratio: float, n_q_tokens: int, n_doc_tokens: int, n_pages: int):
        """Record one (query, ratio) observation."""
        if ratio not in self.data:
            self.data[ratio] = []
        flops = 2 * n_q_tokens * n_doc_tokens * self.D
        self.data[ratio].append({
            'n_q': n_q_tokens, 'n_d': n_doc_tokens,
            'n_pg': n_pages, 'flops': flops,
        })

    def report(self):
        if not self.data:
            print(f"[{self.name}] No efficiency data collected.")
            return
        baseline_flops = None
        if 1.0 in self.data:
            baseline_flops = np.mean([d['flops'] for d in self.data[1.0]])

        print(f"\n{'='*90}")
        print(f"Efficiency Report — {self.name}")
        print(f"{'='*90}")
        hdr = (f"  {'Ratio':<8} {'n':>5} {'avg_Nq':>8} {'std_Nq':>8} "
               f"{'avg_GFLOPs':>12} {'FLOPs_%':>10} {'Speedup':>9}")
        print(hdr)
        print(f"  {'-'*70}")
        for ratio in sorted(self.data.keys()):
            entries = self.data[ratio]
            n       = len(entries)
            nqs     = [d['n_q'] for d in entries]
            avg_nq  = np.mean(nqs)
            std_nq  = np.std(nqs)
            avg_fl  = np.mean([d['flops'] for d in entries])
            gflops  = avg_fl / 1e9
            if baseline_flops and baseline_flops > 0:
                fl_pct  = avg_fl / baseline_flops * 100
                speedup = baseline_flops / avg_fl
                print(f"  {ratio:<8.0%} {n:>5} {avg_nq:>8.1f} {std_nq:>8.1f} "
                      f"{gflops:>12.4f} {fl_pct:>9.1f}% {speedup:>8.2f}×")
            else:
                print(f"  {ratio:<8.0%} {n:>5} {avg_nq:>8.1f} {std_nq:>8.1f} "
                      f"{gflops:>12.4f} {'—':>10} {'—':>9}")

    def to_dataframe(self):
        """Export as pandas DataFrame for CSV / paper tables."""
        rows = []
        baseline_flops = None
        if 1.0 in self.data:
            baseline_flops = np.mean([d['flops'] for d in self.data[1.0]])
        for ratio in sorted(self.data.keys()):
            entries = self.data[ratio]
            n       = len(entries)
            nqs     = [d['n_q'] for d in entries]
            avg_fl  = np.mean([d['flops'] for d in entries])
            row = {
                'method':     self.name,
                'ratio':      ratio,
                'n_queries':  n,
                'avg_Nq':     round(np.mean(nqs), 2),
                'std_Nq':     round(np.std(nqs), 2),
                'avg_Nd':     round(np.mean([d['n_d'] for d in entries]), 0),
                'avg_FLOPs':  round(avg_fl, 0),
                'avg_GFLOPs': round(avg_fl / 1e9, 6),
            }
            if baseline_flops and baseline_flops > 0:
                row['FLOPs_pct'] = round(avg_fl / baseline_flops * 100, 2)
                row['speedup']   = round(baseline_flops / avg_fl, 4)
            else:
                row['FLOPs_pct'] = None
                row['speedup']   = None
            rows.append(row)
        return pd.DataFrame(rows)


# ---------- Throughput Benchmark ----------

class ThroughputBenchmark:
    """
    Measures MaxSim throughput (queries/sec) under realistic batched conditions
    that saturate the GPU, providing empirical efficiency numbers suitable for
    a paper's experimental section.

    Design:
      - Collects representative query vectors at each ratio during eval.
      - For benchmarking: pads queries to uniform length per ratio, batches them,
        and scores against the full document corpus repeatedly.
      - Warmup runs eliminate CUDA cold-start artifacts.
      - Reports queries/sec and ms/query — metrics reviewers expect.

    Usage:
        bench = ThroughputBenchmark()
        # During eval loops:
        bench.collect(ratio=0.5, q_vecs=merged_vecs)
        # After all methods:
        df = bench.run(doc_matrix, doc_mask)
        bench.report(df)
    """

    def __init__(self, device: str = 'cuda'):
        self.device = device
        self.query_pool = {}   # ratio -> list of (N_q, D) tensors

    def collect(self, ratio: float, q_vecs: torch.Tensor):
        """Store a query's reduced vectors for later benchmarking."""
        if ratio not in self.query_pool:
            self.query_pool[ratio] = []
        self.query_pool[ratio].append(q_vecs.detach().to(self.device))

    @torch.no_grad()
    def run(self, doc_matrix, doc_mask,
            n_warmup: int = 10, n_reps: int = 50,
            batch_size: int = 32):
        """
        Run throughput benchmark for all collected ratios.

        Args:
            doc_matrix : (n_docs, max_doc_len, D)
            doc_mask   : (n_docs, max_doc_len)
            n_warmup   : untimed iterations
            n_reps     : timed iterations
            batch_size : queries per batch

        Returns:
            pd.DataFrame with throughput per ratio.
        """
        n_docs      = doc_matrix.shape[0]
        D           = doc_matrix.shape[2]
        total_d_tok = doc_mask.sum().item()
        results     = []

        for ratio in sorted(self.query_pool.keys()):
            pool = self.query_pool[ratio]
            if not pool:
                continue

            n_q_sizes = [q.shape[0] for q in pool]
            max_nq    = max(n_q_sizes)
            avg_nq    = np.mean(n_q_sizes)

            # Pad to uniform N_q and stack
            padded, q_masks = [], []
            for q in pool:
                nq = q.shape[0]
                if nq < max_nq:
                    padded.append(torch.cat([q, torch.zeros(max_nq - nq, D, device=self.device)]))
                else:
                    padded.append(q[:max_nq])
                m = torch.zeros(max_nq, device=self.device, dtype=torch.bool)
                m[:min(nq, max_nq)] = True
                q_masks.append(m)

            q_bank    = torch.stack(padded)
            mask_bank = torch.stack(q_masks)
            pool_size = q_bank.shape[0]
            bs        = min(batch_size, pool_size)

            def _score_batch(idxs):
                for i in idxs:
                    q_valid = q_bank[i][mask_bank[i]]
                    if q_valid.shape[0] == 0:
                        continue
                    sim = torch.einsum('qd,nld->qnl', q_valid, doc_matrix)
                    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
                    sim.max(dim=-1).values.sum(dim=0)

            # Warmup
            for _ in range(n_warmup):
                _score_batch(torch.randint(0, pool_size, (bs,)))
            torch.cuda.synchronize()

            # Timed
            total_q = 0
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_reps):
                idxs = torch.randint(0, pool_size, (bs,))
                _score_batch(idxs)
                total_q += bs
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0

            qps      = total_q / elapsed
            ms_per_q = (elapsed / total_q) * 1000.0
            flops_pq = 2 * avg_nq * total_d_tok * D
            gflops_s = (flops_pq * qps) / 1e9

            results.append({
                'ratio':           ratio,
                'avg_Nq':          round(avg_nq, 1),
                'n_docs':          n_docs,
                'batch_size':      bs,
                'n_reps':          n_reps,
                'total_queries':   total_q,
                'elapsed_s':       round(elapsed, 3),
                'queries_per_sec': round(qps, 1),
                'ms_per_query':    round(ms_per_q, 3),
                'GFLOPs_per_sec':  round(gflops_s, 2),
            })

        return pd.DataFrame(results)

    def report(self, df):
        """Pretty-print throughput benchmark."""
        if df.empty:
            print("No throughput data.")
            return
        print(f"\n{'='*90}")
        print(f"Throughput Benchmark — MaxSim scoring (batched, GPU-saturating)")
        print(f"{'='*90}")
        print(f"  {'Ratio':<8} {'avg_Nq':>8} {'batch':>6} {'reps':>5} "
              f"{'Q/sec':>10} {'ms/query':>10} {'GFLOP/s':>10}")
        print(f"  {'-'*68}")
        for _, row in df.iterrows():
            print(f"  {row['ratio']:<8.0%} {row['avg_Nq']:>8.1f} "
                  f"{row['batch_size']:>6} {row['n_reps']:>5} "
                  f"{row['queries_per_sec']:>10.1f} {row['ms_per_query']:>10.3f} "
                  f"{row['GFLOPs_per_sec']:>10.2f}")


print("Utility functions ready (EfficiencyTracker + ThroughputBenchmark).")


# ==============================================================================
# Spherical KMeans (cosine similarity) — Method 5
# Input : X (N, D) L2-normalized query token vectors
# Output: centroids (K, D) — mean-pool each cluster then re-normalize
# ==============================================================================

def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    """
    Cluster N query token vectors into K representative centroids
    using cosine distance (spherical KMeans).

    Args:
        X      : (N, D)  L2-normalized query token vectors
        K      : int     number of clusters (representative tokens to keep)
        n_iters: int     number of Lloyd-style iterations

    Returns:
        centroids: (K, D)  L2-normalized cluster centroids
    """
    N, D = X.shape
    K = min(K, N)

    if K >= N:
        return X.clone()   # every token is its own representative

    # Initialise: pick K distinct tokens at random
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()   # (K, D)

    for _ in range(n_iters):
        # Assignment: cosine sim = dot product when X is normalised
        sim         = torch.mm(X, centroids.t())   # (N, K)
        cluster_ids = sim.argmax(dim=1)             # (N,)

        # Update: mean-pool members → re-normalize
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = (X[members].mean(dim=0)
                                if members.numel() > 0 else centroids[c])
        centroids = F.normalize(new_centroids, dim=-1)

    return centroids   # (K, D)


# ==============================================================================
# Random Token Pruning — Method 6
# Randomly sample round(M * ratio) content tokens; average over N_RANDOM_SEEDS.
# ==============================================================================

def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask,
                      topk_ratios, n_seeds=N_RANDOM_SEEDS):
    """
    For each ratio r keep k = round(M * r) randomly selected tokens.
    Scores are averaged over `n_seeds` independent random draws to reduce
    variance.

    Args:
        q_norm      : (M, D)   content token vectors, L2-normalized
        content_idx : (M,)     positions of content tokens in the full sequence
                               (unused here; kept for API symmetry)
        doc_matrix  : (n_docs, max_len, D)
        doc_mask    : (n_docs, max_len)  bool
        topk_ratios : list[float]  fractions of tokens to KEEP
        n_seeds     : int  number of random samples to average

    Returns:
        results      : dict {ratio: scores (n_docs,)}
        token_counts : dict {ratio: int}   actual token count used per ratio
    """
    M      = q_norm.shape[0]
    n_docs = doc_matrix.shape[0]
    device = q_norm.device

    if M == 0:
        zero = torch.zeros(n_docs, device=device)
        return {r: zero for r in topk_ratios}, {r: 0 for r in topk_ratios}

    results      = {}
    token_counts = {}

    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        if k == M:
            scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0)
        else:
            accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
            for _ in range(n_seeds):
                perm        = torch.randperm(M, device=device)[:k]
                pruned      = q_norm[perm]          # (k, D)
                accumulated += fast_maxsim(pruned, doc_matrix, doc_mask).sum(dim=0)
            scores = accumulated / n_seeds

        results[ratio]      = scores
        token_counts[ratio] = k

    return results, token_counts

Device: cuda
Utility functions ready (EfficiencyTracker + ThroughputBenchmark).


## Cell 6 — Method 1: Traditional MaxSim (Baseline)

Standard ColPali/ColSMoL retrieval: sum of MaxSim over all query tokens, no pruning.

In [10]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim
# Queries encoded live with ColPali (plain forward, no attention capture).
#
# FIX: Score each query against only its own document's pages (per-document),
# matching the eval notebook. Use set-based recall instead of binary hit.
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_efficiency     = EfficiencyTracker("Traditional MaxSim", D=128)
throughput_bench    = ThroughputBenchmark(device=device)   # shared across all methods

METHOD_KEYS_TRAD = ['traditional']

# Build full doc matrix from all page embeddings (used for per-doc slicing)
print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question  = item['question']
    gt_local  = item['gt_local_indices']       # LOCAL indices within this document
    doc_idxs  = item['doc_embed_indices']       # global embed indices for this doc's pages
    domain    = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    # FIX: Use same tokenization as encoding notebook:
    #   "Query: {question}<pad><pad>...<pad>" via processor() directly
    q_text = f"Query: {question}" + "<pad>" * 10
    q_inputs = query_processor(
        text=[q_text], return_tensors="pt", padding="longest", max_length=600
    ).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)  — plain forward

    attn_mask = q_inputs['attention_mask'][0]                       # (S,)
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()                         # (N, dim)
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Per-document retrieval (matching eval notebook) ─────────────────
    # Slice only this document's pages from the global doc_matrix
    doc_mat  = doc_matrix[doc_idxs]     # (n_doc_pages, max_len, D)
    doc_msk  = doc_mask[doc_idxs]       # (n_doc_pages, max_len)
    n_pages  = len(doc_idxs)

    M      = fast_maxsim(q_norm, doc_mat, doc_msk)               # (N_q, n_pages)
    scores = M.sum(dim=0)                                         # (n_pages,)
    scores_list = scores.cpu().tolist()

    # Track efficiency: N_q tokens, total doc tokens, n_pages
    n_doc_tokens = doc_msk.sum().item()
    trad_efficiency.add(1.0, q_norm.shape[0], n_doc_tokens, n_pages)
    throughput_bench.collect(1.0, q_norm)

    # Evaluate at each K using set-based metrics (matching eval notebook)
    m = hit_metrics(
        top_k_indices(scores_list, max(TOPK_EVAL)),
        gt_local, TOPK_EVAL
    )
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
    }
    for k in TOPK_EVAL:
        row[f'trad_r@{k}']    = round(m[f'r{k}'], 4)
        row[f'trad_ndcg@{k}'] = round(m[f'n{k}'], 4)
    trad_query_rows.append(row)

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_efficiency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

>>> METHOD 1: Traditional MaxSim
Building doc matrix from 20395 page embeddings...
Doc matrix shape: torch.Size([20395, 1131, 128])


Traditional MaxSim:   0%|          | 0/1658 [00:00<?, ?it/s]


Traditional MaxSim Results
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608

Efficiency Report — Traditional MaxSim
  Ratio        n   avg_Nq   std_Nq   avg_GFLOPs    FLOPs_%   Speedup
  ----------------------------------------------------------------------
  100%      1658     41.8      9.5       0.7328     100.0%     1.00×

✅ Saved: traditional_queries.csv


## Cell 7 — Method 2: Ours — SVD Importance + Cluster Pool (v12-Ablation)

This method requires running the model on queries to extract per-token importance scores (SVD on attention). Since we have pre-encoded query embeddings **without** importance, this cell shows two variants:
- **trad_weighted** — uniform token weights (proxy for ablation if importance not available)
- Full ablation requires the model; skip this cell if model is not loaded.

> **Note:** The full ablation (with SVD importance) requires running ColSMoL on query text.
> If you have a pre-computed importance pickle, load it here instead.

### Cell 7a — FIX: SVD helpers + build_cluster_pool_scores

In [11]:
# ==============================================================================
# Cell 7a — FIX: SVD helpers + build_cluster_pool_scores
# ==============================================================================

SVD_RANK_REMOVE  = 1
TEMPERATURE_OURS = 0.5


def remove_sink_components_batch(attn_heads, k):
    """SVD sink removal, batched over (B*H, Sq, Sk)."""
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    """
    attentions   : list of (B, H, Sq, Sk) tensors — one per layer
    content_mask : (B, S) float tensor
    layer_weights: (n_layers,) tensor, exponentially increasing
    Returns      : (B, S) importance tensor
    """
    device_    = content_mask.device
    B, S       = content_mask.shape
    importance = torch.zeros(B, S, device=device_)

    for i, attn in enumerate(attentions):
        attn      = attn.float().to(device_)
        B_, H, Sq, Sk = attn.shape
        attn_flat = attn.view(B_ * H, Sq, Sk)
        cleaned   = remove_sink_components_batch(attn_flat, k)
        cleaned   = cleaned.view(B_, H, Sq, Sk)

        layer_imp = cleaned.sum(dim=2).mean(dim=1)           # (B, S)
        layer_imp = layer_imp * content_mask
        layer_imp = layer_imp / layer_imp.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance += layer_weights[i] * layer_imp

    importance = importance * content_mask
    importance = F.softplus(importance / TEMPERATURE_OURS)
    importance = importance * content_mask
    return importance


def build_content_mask_qwen(inputs, processor):
    """Mask out padding and special tokens from the attention mask."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()

    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()

    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ------------------------------------------------------------------------------
# FIX: Split cluster pooling into two phases for fair latency measurement.
#
# Phase 1 — build_cluster_pool_vectors (NOT timed):
#   Assigns discarded tokens to nearest kept token, computes weighted-average
#   pooled vectors per cluster.  This is analogous to Ward pooling or KMeans
#   clustering in other methods — pre-processing that is excluded from latency.
#
# Phase 2 — fast_maxsim on pooled vectors (TIMED):
#   Runs MaxSim on the pooled centroid vectors, same as Methods 3/4/5/6 run
#   MaxSim on their reduced representations inside the latency window.
#
# Before this fix, build_cluster_pool_scores did BOTH phases inside the latency
# window, making Method 2 appear slower than it actually is at retrieval time.
# ------------------------------------------------------------------------------
def build_cluster_pool_vectors(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    normalize_mode='pre',
):
    """
    Pooling phase ONLY (NOT timed): assign discarded tokens to nearest kept
    token, then create weighted-average pooled vectors per cluster.

    Returns:
      pooled_vecs    : (C, D)  — one pooled vector per cluster
      pooled_weights : (C,)    — total importance weight per cluster
      total_disc_imp : float   — sum of all discarded token importances
    """
    P = q_norm_disc.shape[0]
    D = q_norm_kept.shape[1]

    if P == 0:
        device_ = q_norm_kept.device
        return (torch.zeros(0, D, device=device_),
                torch.zeros(0, device=device_),
                0.0)

    # Assign each discarded token to its nearest kept token (cosine)
    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())   # (P, K)
    cluster_ids = sim_assign.argmax(dim=-1)                 # (P,)

    total_disc_imp = imp_disc.sum().item()

    pooled_vecs    = []
    pooled_weights = []
    for c in cluster_ids.unique():
        members = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w       = imp_disc[members]
        w_sum   = w.sum().clamp(min=1e-8)
        w_norm  = w / w_sum

        # Weighted mean of member vectors
        pool_vec = (q_norm_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)
        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1).squeeze(0)
        pooled_vecs.append(pool_vec)
        pooled_weights.append(w_sum)

    return (torch.stack(pooled_vecs),
            torch.stack(pooled_weights),
            total_disc_imp)


def build_cluster_pool_scores(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    doc_matrix, doc_mask,
    normalize_mode='pre',
):
    """
    Legacy all-in-one function kept for backward compatibility.
    Calls build_cluster_pool_vectors + fast_maxsim internally.
    """
    P      = q_norm_disc.shape[0]
    n_docs = doc_matrix.shape[0]

    if P == 0:
        return torch.zeros(n_docs, device=doc_matrix.device), 0.0

    pooled_vecs, pooled_weights, total_disc_imp = build_cluster_pool_vectors(
        q_norm_kept, q_norm_disc, imp_kept, imp_disc, normalize_mode,
    )

    # Score: MaxSim on pooled vectors
    M_pooled = fast_maxsim(pooled_vecs, doc_matrix, doc_mask)  # (C, n_docs)
    cluster_scores = (M_pooled * pooled_weights.unsqueeze(-1)).sum(dim=0)

    if total_disc_imp > 1e-8:
        cluster_scores = cluster_scores / total_disc_imp

    return cluster_scores, total_disc_imp

# ------------------------------------------------------------------------------
# FIX: Cluster pooling that MERGES pooled centroids back into kept tokens.
#
# The original approach produced K kept + C pooled = K+C vectors for MaxSim.
# Since C ≈ min(P, K), at high ratios K+C ≈ N — no latency reduction.
#
# New approach: each discarded token is assigned to its nearest kept token,
# then the kept token's vector is replaced by the importance-weighted average
# of itself + its assigned discarded tokens.  Output is exactly K vectors,
# always fewer than N, giving monotonic latency improvement.
#
# This is conceptually the same as "absorb discarded information into kept
# representatives" — the kept tokens become enriched cluster centroids.
# ------------------------------------------------------------------------------
def build_merged_pool_vectors(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    normalize_mode='pre',
):
    """
    Merge discarded tokens into their nearest kept tokens.

    Each kept token becomes a weighted average of itself + its assigned
    discarded tokens, weighted by importance.  Output has exactly K vectors.

    Returns:
      merged_vecs    : (K, D)  — merged vectors (same count as kept)
      merged_weights : (K,)    — total importance per merged vector
      total_disc_imp : float   — sum of all discarded token importances
    """
    K = q_norm_kept.shape[0]
    D = q_norm_kept.shape[1]
    P = q_norm_disc.shape[0]
    device_ = q_norm_kept.device

    total_disc_imp = imp_disc.sum().item() if P > 0 else 0.0

    if P == 0:
        # Nothing to merge — kept tokens unchanged
        return q_norm_kept.clone(), imp_kept.clone(), 0.0

    # Assign each discarded token to its nearest kept token (cosine)
    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())   # (P, K)
    cluster_ids = sim_assign.argmax(dim=-1)                 # (P,)

    # Start with kept vectors and their importances
    merged_vecs    = q_norm_kept.clone().float()   # (K, D)
    merged_weights = imp_kept.clone().float()      # (K,)

    # Weighted-sum accumulation: for each kept token, accumulate its
    # assigned discarded tokens weighted by importance
    weighted_sums = q_norm_kept.float() * imp_kept.unsqueeze(-1)   # (K, D)

    for c in cluster_ids.unique():
        members  = (cluster_ids == c).nonzero(as_tuple=True)[0]
        disc_w   = imp_disc[members]                                # weights
        disc_v   = q_norm_disc[members]                             # vectors
        weighted_sums[c] += (disc_v * disc_w.unsqueeze(-1)).sum(dim=0)
        merged_weights[c] += disc_w.sum()

    # Normalize to get weighted average, then L2-normalize
    merged_vecs = weighted_sums / merged_weights.unsqueeze(-1).clamp(min=1e-8)
    if normalize_mode == 'pre':
        merged_vecs = F.normalize(merged_vecs, dim=-1)

    return merged_vecs, merged_weights, total_disc_imp


def make_ablation_keys_ours():
    keys = []
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad")
                keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS_OURS = make_ablation_keys_ours()

print("✅ Fixed SVD helpers + build_cluster_pool_scores loaded.")

✅ Fixed SVD helpers + build_cluster_pool_scores loaded.


### Cell 7b — FLOPs-based Efficiency Tracking for Method 2 (Ours)

In [12]:
# ==============================================================================
# Cell 7b — FLOPs-based Efficiency Tracking for Method 2 (Ours)
#
# Instead of timing fast_maxsim calls (which are too fast to benchmark
# reliably on small per-document query sets), we track the THEORETICAL
# FLOPs and ACTUAL token counts for each (query, ratio) pair.
#
# This gives a deterministic, reproducible, noise-free efficiency metric
# that scales correctly with the token retention ratio.
#
# For Method 2 specifically, the effective N_q at each ratio is:
#   all_q_pre = cat([q_kept, pooled_centroids])
#   N_q = n_keep + n_unique_clusters
# which may differ from round(N * r) due to the cluster merging step.
# ==============================================================================

print(">>> EFFICIENCY TRACKING — Ours (SVD+ClusterPool)")

# Use the first config as representative (same as old latency measurement)
EFF_N_LAYERS = N_LAST_LAYERS_LIST[0]
EFF_MODE     = NORMALIZE_MODES[0]

print(f"  Config: n_layers={EFF_N_LAYERS}, mode={EFF_MODE}")

ours_efficiency = EfficiencyTracker("Ours (SVD+ClusterPool)", D=128)

for q_idx, item in tqdm(enumerate(qa_pairs[:100]), total=len(qa_pairs[:100]), desc="Ours Efficiency"):
    question = item['question']
    doc_idxs = item['doc_embed_indices']

    proj, raw_outputs, q_inputs, _ = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d    = q_inputs['attention_mask'][0].float()
    content_mask_1d = build_content_mask_qwen(q_inputs, query_processor)[0].float()

    trad_idx   = torch.where(attn_mask_1d   > 0)[0]
    method_idx = torch.where(content_mask_1d > 0)[0]
    if trad_idx.numel() == 0:
        continue
    if method_idx.numel() == 0:
        method_idx = trad_idx

    content_mask_2d = content_mask_1d.unsqueeze(0)
    all_attns       = raw_outputs.attentions

    if all_attns is not None and len(all_attns) > 0:
        attn_list = list(all_attns[-EFF_N_LAYERS:])
        n_actual  = len(attn_list)
    else:
        attn_list = []
        n_actual  = 0

    layer_weights = torch.exp(
        torch.linspace(0, 1, max(n_actual, 1), device=device)
    )
    layer_weights /= layer_weights.sum()

    if n_actual > 0:
        importance = compute_svd_importance_softplus(
            attn_list, content_mask_2d, layer_weights, SVD_RANK_REMOVE
        )
    else:
        importance = content_mask_2d.clone()

    importance = (importance * content_mask_2d)[0]
    q_embed    = proj[0].float()

    q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)
    imp_valid      = importance[method_idx].float()
    n_method       = method_idx.numel()
    sorted_imp_idx = torch.argsort(imp_valid, descending=True)

    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]
    n_doc_tokens = doc_msk.sum().item()
    n_pages      = len(doc_idxs)

    # Record full-token baseline
    ours_efficiency.add(1.0, n_method, n_doc_tokens, n_pages)
    throughput_bench.collect(1.0, q_method_norm)

    for r in TOPK_RATIOS:
        n_keep   = max(1, int(n_method * r))
        kept_idx = sorted_imp_idx[:n_keep]
        disc_idx = sorted_imp_idx[n_keep:]

        q_kept   = q_method_norm[kept_idx]
        q_disc   = q_method_norm[disc_idx]
        imp_kept = imp_valid[kept_idx]
        imp_disc = imp_valid[disc_idx]

        P = q_disc.shape[0]
        if P > 0:
            sim_assign  = torch.mm(q_disc, q_kept.t())
            cluster_ids = sim_assign.argmax(dim=-1)
            unique_clusters = cluster_ids.unique()
            n_pooled    = unique_clusters.shape[0]

            pooled_raw = []
            for c in unique_clusters:
                members = (cluster_ids == c).nonzero(as_tuple=True)[0]
                w       = imp_disc[members]
                w_sum   = w.sum().clamp(min=1e-8)
                w_norm  = w / w_sum
                pv      = (q_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)
                pooled_raw.append(pv)
            pooled_pre = F.normalize(torch.stack(pooled_raw), dim=-1)
            all_q      = torch.cat([q_kept, pooled_pre], dim=0)
        else:
            all_q = q_kept

        # Record: N_q = n_keep + n_pooled_clusters (the ACTUAL vectors fed to MaxSim)
        ours_efficiency.add(r, all_q.shape[0], n_doc_tokens, n_pages)
        throughput_bench.collect(r, all_q)

    if q_idx % 200 == 0:
        gc.collect()
        torch.cuda.empty_cache()

ours_efficiency.report()
print("\n✅ Ours efficiency tracking complete.")

>>> EFFICIENCY TRACKING — Ours (SVD+ClusterPool)
  Config: n_layers=4, mode=pre


Ours Efficiency:   0%|          | 0/100 [00:00<?, ?it/s]


Efficiency Report — Ours (SVD+ClusterPool)
  Ratio        n   avg_Nq   std_Nq   avg_GFLOPs    FLOPs_%   Speedup
  ----------------------------------------------------------------------
  10%        100      7.9      2.2       0.0713      17.6%     5.68×
  20%        100     15.5      4.4       0.1402      34.6%     2.89×
  30%        100     21.6      6.0       0.1951      48.2%     2.08×
  40%        100     26.9      7.4       0.2438      60.2%     1.66×
  50%        100     32.0      8.5       0.2892      71.4%     1.40×
  60%        100     36.1      9.6       0.3269      80.7%     1.24×
  70%        100     39.5     10.2       0.3577      88.3%     1.13×
  80%        100     42.4     11.0       0.3844      94.9%     1.05×
  90%        100     44.2     11.1       0.4006      98.9%     1.01×
  100%       100     44.7     11.1       0.4049     100.0%     1.00×

✅ Ours efficiency tracking complete.


### Cell 7c — METHOD 2: Ours — SVD Importance + Cluster Pool [FAST]

In [13]:
# ==============================================================================
# Cell 7c — METHOD 2 OPTIMIZED: SVD Importance + Cluster Pool
#
# Key speed-ups vs original:
#   1. Attention hooks registered on ONLY the last max(N_LAST_LAYERS_LIST)
#      layers — avoids capturing & storing all 32 layers every query.
#   2. Importance computed ONCE per n_layers variant (not re-derived inside
#      the mode/ratio loop).
#   3. Sort order computed ONCE per n_layers variant; shared across all modes
#      and ratios.
#   4. Cluster pooling (build_cluster_pool_vectors) computed ONCE per
#      (n_layers, ratio) — mode is applied AFTER clustering, so we split
#      the pooled vec normalization out instead of re-running clustering per mode.
#   5. MaxSim called ONCE per (n_layers, ratio) on the combined
#      (kept + pooled) vectors — both scoring variants (trad, weighted) are
#      derived from that single M_all call.
#   6. gc.collect / empty_cache frequency reduced to every 200 queries
#      (GC dominates runtime at every-50 cadence for small models).
# ==============================================================================

print(">>> METHOD 2 OPTIMIZED: Ours — SVD Importance + Cluster Pool")

ours_metrics        = {}
ours_domain_metrics = {}
ours_query_rows     = []

print(f"Running ablation over {len(qa_pairs)} queries")
print(f"  n_last_layers : {N_LAST_LAYERS_LIST}")
print(f"  normalize_mode: {NORMALIZE_MODES}")
print(f"  topk_ratios   : {TOPK_RATIOS}")

# ── Speed-up 1: only capture the last N layers we actually need ──────────────
MAX_LAYERS_NEEDED = max(N_LAST_LAYERS_LIST)

def encode_query_optimized(question, processor, model, device):
    """
    Like encode_query_live but hooks only the last MAX_LAYERS_NEEDED layers,
    cutting hook overhead by up to 8x (32 → 4 layers for our config).
    No latency tracking (caller doesn't need it).
    
    FIX: Uses same tokenization as encoding notebook:
      "Query: {question}<pad><pad>...<pad>" via processor() directly
    """
    q_text = f"Query: {question}" + "<pad>" * 10
    inputs = processor(
        text=[q_text], return_tensors="pt", padding="longest", max_length=600
    ).to(device)
    collector = AttentionCollectorPage()
    layers    = _find_text_layers_page(model)
    n_total   = len(layers)
    # Register only the layers we will actually slice into
    hook_indices = list(range(n_total - MAX_LAYERS_NEEDED, n_total))
    collector.register_hooks(model, hook_indices)
    try:
        with torch.no_grad():
            kwargs = dict(inputs)
            kwargs['output_attentions'] = True
            proj = model(**kwargs)
    finally:
        collector.remove_hooks()

    attns = tuple(collector.attentions)
    if len(attns) == 0:
        print("⚠️  WARNING: no attention tensors captured — check attn_implementation='eager'")
    raw_outputs = SimpleNamespace(attentions=attns if attns else None)
    collector.clear()
    return proj, raw_outputs, inputs


pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Ours (SVD+ClusterPool) OPT")

for q_idx, item in pbar:
    question = item['question']
    gt_local = item['gt_local_indices']
    doc_idxs = item['doc_embed_indices']
    domain   = item['domain']

    # ── Encode (hooks on last MAX_LAYERS_NEEDED layers only) ────────────────
    proj, raw_outputs, q_inputs = encode_query_optimized(
        question, query_processor, query_model, device
    )

    attn_mask_1d    = q_inputs['attention_mask'][0].float()
    content_mask_1d = build_content_mask_qwen(q_inputs, query_processor)[0].float()

    trad_idx   = torch.where(attn_mask_1d   > 0)[0]
    method_idx = torch.where(content_mask_1d > 0)[0]
    if trad_idx.numel() == 0:
        continue
    if method_idx.numel() == 0:
        method_idx = trad_idx

    content_mask_2d = content_mask_1d.unsqueeze(0)
    all_attns       = raw_outputs.attentions   # tuple of MAX_LAYERS_NEEDED tensors

    # Per-document scoring
    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]
    n_pages = len(doc_idxs)

    # ── Speed-up 2: compute importance ONCE per n_layers variant ────────────
    embed_cache = {}

    for n_layers in N_LAST_LAYERS_LIST:
        if all_attns is not None and len(all_attns) > 0:
            attn_list = list(all_attns[-n_layers:])
            n_actual  = len(attn_list)
        else:
            attn_list = []
            n_actual  = 0

        layer_weights = torch.exp(
            torch.linspace(0, 1, max(n_actual, 1), device=device)
        )
        layer_weights /= layer_weights.sum()

        if n_actual > 0:
            importance = compute_svd_importance_softplus(
                attn_list, content_mask_2d, layer_weights, SVD_RANK_REMOVE
            )
        else:
            importance = content_mask_2d.clone()

        importance = (importance * content_mask_2d)[0]

        q_embed        = proj[0].float()
        q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)
        imp_valid      = importance[method_idx].float()
        sorted_imp_idx = torch.argsort(imp_valid, descending=True)

        embed_cache[n_layers] = (q_method_norm, imp_valid, sorted_imp_idx)

    query_row = {
        'query_id': q_idx,
        'doc_name': item['doc_name'],
        'domain':   domain,
        'question': question,
    }

    # ── Ablation loop ────────────────────────────────────────────────────────
    for n_layers in N_LAST_LAYERS_LIST:
        q_method_norm, imp_valid, sorted_imp_idx = embed_cache[n_layers]
        n_method   = method_idx.numel()
        imp_sum    = imp_valid.sum().clamp(min=1e-8)

        for r in TOPK_RATIOS:
            n_keep   = max(1, int(n_method * r))
            kept_idx = sorted_imp_idx[:n_keep]
            disc_idx = sorted_imp_idx[n_keep:]

            q_kept   = q_method_norm[kept_idx]
            q_disc   = q_method_norm[disc_idx]
            imp_kept = imp_valid[kept_idx]
            imp_disc = imp_valid[disc_idx]

            P = q_disc.shape[0]
            if P > 0:
                sim_assign  = torch.mm(q_disc, q_kept.t())
                cluster_ids = sim_assign.argmax(dim=-1)
                total_disc_imp = imp_disc.sum().item()
                pool_frac = total_disc_imp / imp_sum.item()

                unique_clusters = cluster_ids.unique()
                pooled_raw   = []
                pooled_w     = []
                for c in unique_clusters:
                    members = (cluster_ids == c).nonzero(as_tuple=True)[0]
                    w       = imp_disc[members]
                    w_sum   = w.sum().clamp(min=1e-8)
                    w_norm  = w / w_sum
                    pv      = (q_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)
                    pooled_raw.append(pv)
                    pooled_w.append(w_sum)

                pooled_raw = torch.stack(pooled_raw)
                pooled_w   = torch.stack(pooled_w)
            else:
                total_disc_imp = 0.0
                pool_frac      = 0.0
                pooled_raw     = None
                pooled_w       = None

            if pooled_raw is not None:
                pooled_pre  = F.normalize(pooled_raw, dim=-1)
                all_q_pre   = torch.cat([q_kept, pooled_pre], dim=0)
            else:
                all_q_pre   = q_kept

            M_all   = fast_maxsim(all_q_pre, doc_mat, doc_msk)
            M_kept  = M_all[:n_keep]
            M_pool  = M_all[n_keep:]

            if pooled_raw is not None and M_pool.shape[0] > 0:
                pool_contrib = (M_pool * pooled_w.unsqueeze(-1)).sum(dim=0)
                if total_disc_imp > 1e-8:
                    pool_contrib = pool_contrib / total_disc_imp
                pool_contrib = pool_contrib * pool_frac
            else:
                pool_contrib = torch.zeros(n_pages, device=device)

            EXACT_POST = False

            for mode in NORMALIZE_MODES:
                tag = f"L{n_layers}_norm{mode}_r{int(r*100)}"

                if mode == 'post' and EXACT_POST and pooled_raw is not None:
                    pass

                # Traditional scoring
                scores_trad  = (M_kept.sum(dim=0) + pool_contrib).cpu().tolist()
                m_trad   = hit_metrics(top_k_indices(scores_trad, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
                key_trad = f"{tag}_trad"
                record(ours_metrics, ours_domain_metrics, key_trad, m_trad, domain)
                for kk in TOPK_EVAL:
                    query_row[f'{key_trad}_r@{kk}']    = round(m_trad[f'r{kk}'], 4)
                    query_row[f'{key_trad}_ndcg@{kk}'] = round(m_trad[f'n{kk}'], 4)

                # Weighted scoring
                imp_kept_n   = imp_kept / imp_kept.sum().clamp(min=1e-8)
                scores_wt    = ((M_kept * imp_kept_n.unsqueeze(-1)).sum(dim=0) + pool_contrib).cpu().tolist()
                m_wt   = hit_metrics(top_k_indices(scores_wt, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
                key_wt = f"{tag}_weighted"
                record(ours_metrics, ours_domain_metrics, key_wt, m_wt, domain)
                for kk in TOPK_EVAL:
                    query_row[f'{key_wt}_r@{kk}']    = round(m_wt[f'r{kk}'], 4)
                    query_row[f'{key_wt}_ndcg@{kk}'] = round(m_wt[f'n{kk}'], 4)

    ours_query_rows.append(query_row)

    # Speed-up 5: GC every 200 queries instead of every 50
    if q_idx % 200 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print_summary(ours_metrics, ours_domain_metrics,
              ABLATION_KEYS_OURS,
              title="Ours — SVD Importance + Cluster Pool [OPTIMIZED]")

pd.DataFrame(ours_query_rows).to_csv(
    os.path.join(WORKING_DIR, "ours_ablation_queries.csv"), index=False)
print("\n✅ Saved: ours_ablation_queries.csv")

>>> METHOD 2 OPTIMIZED: Ours — SVD Importance + Cluster Pool
Running ablation over 1658 queries
  n_last_layers : [4, 8, 16, 32]
  normalize_mode: ['pre', 'post']
  topk_ratios   : [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


Ours (SVD+ClusterPool) OPT:   0%|          | 0/1658 [00:00<?, ?it/s]

[AttentionCollector] Using layer stack: 'base_model.model.model.text_model.layers' (32 layers)

Ours — SVD Importance + Cluster Pool [OPTIMIZED]
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
L4_normpre_r10_trad                  51.14%  74.79%  82.42%  88.74%   0.5700   0.6736   0.7058   0.7297
L4_normpre_r10_weighted              53.57%  76.18%  82.73%  88.97%   0.5983   0.6936   0.7209   0.7442
L4_normpre_r20_trad                  57.01%  76.98%  83.56%  90.64%   0.6333   0.7121   0.7402   0.7662
L4_normpre_r20_weighted              57.26%  78.32%  83.77%  90.16%   0.6369   0.7218   0.7451   0.7688
L4_normpre_r30_trad                  56.87%  77.80%  84.15%  90.60%   0.6327   0.7176   0.7448   0.7684
L4_normpre_r30_weighted              57.23%  78.34%  83.73%  90.66%   0.6363   0.7225   0.7455   0.7708
L4_normpre_r40_trad    

## Cell 8 — Method 3: Hierarchical (Ward Agglomerative Token Pooling)

In [14]:
# ==============================================================================
# METHOD 3 — Hierarchical: Ward agglomerative token pooling
# ==============================================================================

print(">>> METHOD 3: Hierarchical Ward Token Pooling")


def _ward_distance_matrix(cents, sizes):
    device_  = cents.device
    sim      = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist  = 2.0 * (1.0 - sim)
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    mask = torch.ones(cents.shape[0], cents.shape[0], dtype=torch.bool, device=device_).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    """vecs: (N, D) L2-normalized. Returns centroids (K, D) L2-normalized."""
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()

    device_ = vecs.device
    sums    = vecs.clone().float()
    sizes   = torch.ones(N, device=device_, dtype=torch.float32)
    cents   = F.normalize(sums, dim=-1)
    active  = torch.ones(N, dtype=torch.bool, device=device_)
    C       = N

    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act      = cents[active_idx]
        s_act      = sizes[active_idx]
        ward       = _ward_distance_matrix(c_act, s_act)
        flat_idx   = ward.argmin().item()
        n_act      = active_idx.shape[0]
        ai, aj     = flat_idx // n_act, flat_idx % n_act
        gi, gj     = active_idx[ai].item(), active_idx[aj].item()
        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1

    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]


def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    """
    Returns (results, token_counts) where:
      results      : dict[ratio -> scores tensor]
      token_counts : dict[ratio -> int]  actual centroid count after pooling
    """
    N            = q_norm.shape[0]
    results      = {}
    token_counts = {}
    prev_vecs, prev_C = q_norm.clone(), N

    for ratio in sorted(topk_ratios, reverse=True):
        target_C = max(1, round(N * ratio))
        if target_C >= prev_C:
            centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C)
            prev_vecs, prev_C = centroids, centroids.shape[0]

        M_c            = fast_maxsim(centroids, doc_matrix, doc_mask)
        results[ratio] = M_c.sum(0)
        token_counts[ratio] = centroids.shape[0]

    return results, token_counts


# ---------- Keys ----------
METHOD_KEYS_HIER = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]

hier_metrics        = {}
hier_domain_metrics = {}
hier_query_rows     = []
hier_efficiency     = EfficiencyTracker("Hierarchical Ward Pool", D=128)

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Hierarchical Ward Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_local = item['gt_local_indices']
    doc_idxs = item['doc_embed_indices']
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    # FIX: Use same tokenization as encoding notebook
    q_text = f"Query: {question}" + "<pad>" * 10
    q_inputs = query_processor(
        text=[q_text], return_tensors="pt", padding="longest", max_length=600
    ).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # plain forward

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Per-document retrieval ─────────────────────────────────────────
    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]

    # Baseline traditional
    M_trad     = fast_maxsim(q_norm, doc_mat, doc_msk)
    trad_sc    = M_trad.sum(dim=0).cpu().tolist()
    m_trad     = hit_metrics(top_k_indices(trad_sc, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
    record(hier_metrics, hier_domain_metrics, 'traditional', m_trad, domain)

    # Ward hierarchical pooling — all ratios
    ratio_scores, ratio_token_counts = ward_pool_scores_all_ratios(q_norm, doc_mat, doc_msk, TOPK_RATIOS)
    n_doc_tokens = doc_msk.sum().item()
    n_pages      = len(doc_idxs)

    # Record full-token baseline for FLOPs% and Speedup calculation
    hier_efficiency.add(1.0, q_norm.shape[0], n_doc_tokens, n_pages)

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
    }
    for k in TOPK_EVAL:
        query_row[f'trad_r@{k}']    = round(m_trad[f'r{k}'], 4)
        query_row[f'trad_ndcg@{k}'] = round(m_trad[f'n{k}'], 4)

    for r in TOPK_RATIOS:
        key    = f"hier_r{int(r*100)}"
        sc     = ratio_scores[r].cpu().tolist()
        m      = hit_metrics(top_k_indices(sc, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
        record(hier_metrics, hier_domain_metrics, key, m, domain)
        hier_efficiency.add(r, ratio_token_counts[r], n_doc_tokens, n_pages)
        for kk in TOPK_EVAL:
            query_row[f'{key}_r@{kk}']    = round(m[f'r{kk}'], 4)
            query_row[f'{key}_ndcg@{kk}'] = round(m[f'n{kk}'], 4)

    hier_query_rows.append(query_row)

print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER,
              title="Hierarchical Ward Pooling Results")
hier_efficiency.report()

pd.DataFrame(hier_query_rows).to_csv(
    os.path.join(WORKING_DIR, "hierarchical_queries.csv"), index=False)
print("\n✅ Saved: hierarchical_queries.csv")

>>> METHOD 3: Hierarchical Ward Token Pooling


Hierarchical Ward Pool:   0%|          | 0/1658 [00:00<?, ?it/s]


Hierarchical Ward Pooling Results
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608
hier_r10                             45.53%  68.84%  76.45%  86.15%   0.5066   0.6119   0.6447   0.6791
hier_r20                             52.82%  75.20%  82.69%  89.54%   0.5881   0.6819   0.7137   0.7389
hier_r30                             55.35%  77.11%  83.84%  90.85%   0.6176   0.7065   0.7345   0.7603
hier_r40                             56.34%  78.22%  84.12%  90.86%   0.6242   0.7166   0.7418   0.7668
hier_r50                             56.63%  78.37%  84.41%  91.17%   0.6285   0.7193   0.7448   0.7699
hier_r60                             57.10%  78.06%  84.59%  91.33%   0.6345   0.7203   0.7482   0.7728
hier_r70                     

## Cell 9 — Method 4: Attention Score Token Pruning (Lassance et al. 2021)

In [15]:
# ==============================================================================
# METHOD 4 — Attention Score Token Pruning (v13)
#
# Per-token importance = column-sum of attention weights across all heads/layers.
# Queries are encoded live with forward_with_attentions; latency is measured.
# ==============================================================================

print(">>> METHOD 4: Attention Score Token Pruning (Lassance et al. 2021)")

# ---------- Keys ----------
METHOD_KEYS_ATTN = ['traditional']
for n in ATTN_N_LAYERS_LIST:
    for r in TOPK_RATIOS:
        METHOD_KEYS_ATTN += [
            f"attn_L{n}_r{int(r*100)}_trad",
            f"attn_L{n}_r{int(r*100)}_weighted",
        ]

attn_metrics        = {}
attn_domain_metrics = {}
attn_query_rows     = []
attn_efficiency     = EfficiencyTracker("Attention Score Pruning", D=128)

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Attention Score Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_local = item['gt_local_indices']
    doc_idxs = item['doc_embed_indices']
    domain   = item['domain']

    # ── Encode query live (with attentions) ───────────────────────────
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(
        question, query_processor, query_model, device
    )

    attn_mask_1d = q_inputs['attention_mask'][0].float()
    trad_idx     = torch.where(attn_mask_1d > 0)[0]
    N            = trad_idx.numel()

    q_emb  = proj[0][trad_idx].float()
    q_norm = F.normalize(q_emb, dim=-1)

    # Per-document scoring
    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]

    # Baseline (full tokens)
    M_full     = fast_maxsim(q_norm, doc_mat, doc_msk)
    trad_sc    = M_full.sum(dim=0).cpu().tolist()
    m_trad     = hit_metrics(top_k_indices(trad_sc, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
    record(attn_metrics, attn_domain_metrics, 'traditional', m_trad, domain)

    # Record full-token baseline for FLOPs% and Speedup calculation
    n_doc_tokens_attn = doc_msk.sum().item()
    attn_efficiency.add(1.0, q_norm.shape[0], n_doc_tokens_attn, len(doc_idxs))

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
    }
    for k in TOPK_EVAL:
        query_row[f'trad_r@{k}']    = round(m_trad[f'r{k}'], 4)
        query_row[f'trad_ndcg@{k}'] = round(m_trad[f'n{k}'], 4)

    for n_layers in ATTN_N_LAYERS_LIST:
        # Compute attention column-sum importance from the last n_layers attention tensors
        # i(t) = Σ_h Σ_j A_{h,j,t}  (sum of attention received by token t, Lassance 2021)
        if raw_outputs.attentions is not None and len(raw_outputs.attentions) >= 1:
            attn_subset = list(raw_outputs.attentions[-n_layers:])
            # Each element: (1, H, Sq, Sk)  → sum over query dim → (1, H, Sk) → mean heads → (1, Sk)
            imp_2d = torch.zeros(1, q_inputs['attention_mask'].shape[1], device=device)
            for attn_layer in attn_subset:
                col_sum = attn_layer.float().sum(dim=2).mean(dim=1)  # (1, S)
                imp_2d += col_sum
            imp_full = imp_2d[0]                                      # (S,)
            imp      = imp_full[trad_idx]                             # (N,)
        else:
            imp = torch.ones(N, device=device)

        sorted_imp_idx = torch.argsort(imp, descending=True)

        for r in TOPK_RATIOS:
            n_keep   = max(1, int(N * r))

            # ── Pruning (NOT timed) ────────────────────────────
            kept_idx = sorted_imp_idx[:n_keep]
            imp_kept = imp[kept_idx]
            q_kept   = q_norm[kept_idx]                          # pruned vectors

            M_kept   = fast_maxsim(q_kept, doc_mat, doc_msk)
            sc_trad  = M_kept.sum(dim=0).cpu().tolist()
            top_t    = top_k_indices(sc_trad, max(TOPK_EVAL))

            imp_k_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
            sc_w     = (M_kept * imp_k_n.unsqueeze(-1)).sum(dim=0).cpu().tolist()
            top_w    = top_k_indices(sc_w, max(TOPK_EVAL))

            n_doc_tokens_attn = doc_msk.sum().item()
            attn_efficiency.add(r, q_kept.shape[0], n_doc_tokens_attn, len(doc_idxs))
            # ────────────────────────────────────────────────────

            m_t      = hit_metrics(top_t, gt_local, TOPK_EVAL)
            key_t    = f"attn_L{n_layers}_r{int(r*100)}_trad"
            record(attn_metrics, attn_domain_metrics, key_t, m_t, domain)
            for kk in TOPK_EVAL:
                query_row[f'{key_t}_r@{kk}']    = round(m_t[f'r{kk}'], 4)
                query_row[f'{key_t}_ndcg@{kk}'] = round(m_t[f'n{kk}'], 4)

            m_w      = hit_metrics(top_w, gt_local, TOPK_EVAL)
            key_w    = f"attn_L{n_layers}_r{int(r*100)}_weighted"
            record(attn_metrics, attn_domain_metrics, key_w, m_w, domain)
            for kk in TOPK_EVAL:
                query_row[f'{key_w}_r@{kk}']    = round(m_w[f'r{kk}'], 4)
                query_row[f'{key_w}_ndcg@{kk}'] = round(m_w[f'n{kk}'], 4)

    attn_query_rows.append(query_row)

print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1, trad scoring)")
attn_efficiency.report()

pd.DataFrame(attn_query_rows).to_csv(
    os.path.join(WORKING_DIR, "attention_queries.csv"), index=False)
print("\n✅ Saved: attention_queries.csv")

>>> METHOD 4: Attention Score Token Pruning (Lassance et al. 2021)


Attention Score Pruning:   0%|          | 0/1658 [00:00<?, ?it/s]


Attention Score Pruning (L=1, trad scoring)
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608
attn_L1_r10_trad                     46.22%  67.76%  76.42%  85.38%   0.5151   0.6089   0.6460   0.6782
attn_L1_r20_trad                     53.38%  74.31%  81.97%  88.99%   0.5923   0.6785   0.7116   0.7374
attn_L1_r30_trad                     55.38%  77.21%  83.37%  90.30%   0.6158   0.7077   0.7338   0.7589
attn_L1_r40_trad                     56.40%  77.79%  83.92%  90.10%   0.6273   0.7153   0.7413   0.7642
attn_L1_r50_trad                     56.56%  78.09%  84.29%  90.40%   0.6273   0.7181   0.7439   0.7669
attn_L1_r60_trad                     56.65%  78.11%  84.00%  90.37%   0.6285   0.7180   0.7429   0.7665
attn_L1_r70_trad   

## Cell 10 — Method 5: Spherical KMeans Token Pooling

In [16]:
# ==============================================================================
# METHOD 5 — Spherical KMeans Token Pooling
#
# Groups the N query tokens into K = max(1, int(N * ratio)) clusters using
# spherical KMeans (cosine distance).  Each cluster is represented by its
# L2-normalised mean-pooled centroid.  MaxSim is then run with K centroid
# vectors instead of N raw token vectors.
#
# References: PLAID / ColBERT v2 cluster-pruning idea adapted for query-side.
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 5: Spherical KMeans Token Pooling")

# ---------- Keys ----------
METHOD_KEYS_KMEANS = ['traditional'] + [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]

kmeans_metrics        = {}
kmeans_domain_metrics = {}
kmeans_query_rows     = []
kmeans_efficiency     = EfficiencyTracker("Spherical KMeans Pool", D=128)

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Spherical KMeans Pool")

for q_idx, item in pbar:
    question = item['question']
    gt_local = item['gt_local_indices']
    doc_idxs = item['doc_embed_indices']
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    # FIX: Use same tokenization as encoding notebook
    q_text = f"Query: {question}" + "<pad>" * 10
    q_inputs = query_processor(
        text=[q_text], return_tensors="pt", padding="longest", max_length=600
    ).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)          # (N, D)
    N         = q_norm.shape[0]

    # Per-document scoring
    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_mat, doc_msk)
    trad_sc    = M_full.sum(dim=0).cpu().tolist()
    m_trad     = hit_metrics(top_k_indices(trad_sc, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
    record(kmeans_metrics, kmeans_domain_metrics, 'traditional', m_trad, domain)

    # Record full-token baseline for FLOPs% and Speedup calculation
    n_doc_tokens_km = doc_msk.sum().item()
    kmeans_efficiency.add(1.0, q_norm.shape[0], n_doc_tokens_km, len(doc_idxs))

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
    }
    for k in TOPK_EVAL:
        query_row[f'trad_r@{k}']    = round(m_trad[f'r{k}'], 4)
        query_row[f'trad_ndcg@{k}'] = round(m_trad[f'n{k}'], 4)

    # ── Spherical KMeans pooling — all ratios ─────────────────────────────
    for r in TOPK_RATIOS:
        K = max(1, int(N * r))

        # ── KMeans pooling (NOT timed) ────────────────────────────
        centroids = spherical_kmeans(q_norm, K)       # (K, D)

        M_k    = fast_maxsim(centroids, doc_mat, doc_msk)
        scores = M_k.sum(dim=0).cpu().tolist()

        n_doc_tokens_km = doc_msk.sum().item()
        kmeans_efficiency.add(r, centroids.shape[0], n_doc_tokens_km, len(doc_idxs))
        # ────────────────────────────────────────────────────────────

        key = f"kmeans_r{int(r*100)}"
        m   = hit_metrics(top_k_indices(scores, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
        record(kmeans_metrics, kmeans_domain_metrics, key, m, domain)
        for kk in TOPK_EVAL:
            query_row[f'{key}_r@{kk}']    = round(m[f'r{kk}'], 4)
            query_row[f'{key}_ndcg@{kk}'] = round(m[f'n{kk}'], 4)

    kmeans_query_rows.append(query_row)

print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS,
              title="Spherical KMeans Pooling Results")
kmeans_efficiency.report()

pd.DataFrame(kmeans_query_rows).to_csv(
    os.path.join(WORKING_DIR, "kmeans_queries.csv"), index=False)
print("\n✅ Saved: kmeans_queries.csv")

>>> METHOD 5: Spherical KMeans Token Pooling


Spherical KMeans Pool:   0%|          | 0/1658 [00:00<?, ?it/s]


Spherical KMeans Pooling Results
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608
kmeans_r10                           52.26%  73.66%  81.30%  89.07%   0.5802   0.6727   0.7052   0.7334
kmeans_r20                           54.12%  77.09%  83.66%  89.48%   0.6019   0.7012   0.7290   0.7505
kmeans_r30                           55.28%  76.90%  82.75%  89.63%   0.6146   0.7050   0.7295   0.7550
kmeans_r40                           55.86%  76.80%  83.12%  89.71%   0.6212   0.7075   0.7342   0.7587
kmeans_r50                           55.77%  77.63%  83.36%  90.01%   0.6194   0.7127   0.7371   0.7611
kmeans_r60                           55.75%  77.30%  83.37%  89.53%   0.6212   0.7098   0.7353   0.7581
kmeans_r70                    

## Cell 11 — Method 6: Random Token Pruning (Ablation Baseline)


In [17]:
# ==============================================================================
# METHOD 6 — Random Token Pruning (v14)
#
# Ablation baseline: instead of using any importance signal, tokens are sampled
# UNIFORMLY AT RANDOM.  If random pruning performs comparably to attention-based
# or SVD-based pruning, the importance signal offers little value.
#
# For each (query, ratio) pair: sample round(N * ratio) content tokens
# N_RANDOM_SEEDS times and average the resulting MaxSim scores to reduce
# variance from the random draw.
#
# Queries are encoded live with the plain forward pass (no attentions needed).
# ==============================================================================

print(">>> METHOD 6: Random Token Pruning (Ablation Baseline)")

# ---------- Keys ----------
METHOD_KEYS_RAND = ['traditional'] + [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]

rand_metrics        = {}
rand_domain_metrics = {}
rand_query_rows     = []
rand_efficiency     = EfficiencyTracker("Random Pruning", D=128)

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Random Pruning")

for q_idx, item in pbar:
    question = item['question']
    gt_local = item['gt_local_indices']
    doc_idxs = item['doc_embed_indices']
    domain   = item['domain']

    # ── Encode query live (plain forward — no attentions needed) ──────────
    # FIX: Use same tokenization as encoding notebook
    q_text = f"Query: {question}" + "<pad>" * 10
    q_inputs = query_processor(
        text=[q_text], return_tensors="pt", padding="longest", max_length=600
    ).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)   # (N, D) — content tokens only

    # Per-document scoring
    doc_mat = doc_matrix[doc_idxs]
    doc_msk = doc_mask[doc_idxs]

    # ── Baseline (full tokens) ────────────────────────────────────────────
    M_full     = fast_maxsim(q_norm, doc_mat, doc_msk)
    trad_sc    = M_full.sum(dim=0).cpu().tolist()
    m_trad     = hit_metrics(top_k_indices(trad_sc, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
    record(rand_metrics, rand_domain_metrics, 'traditional', m_trad, domain)

    # Record full-token baseline for FLOPs% and Speedup calculation
    n_doc_tokens_rand = doc_msk.sum().item()
    rand_efficiency.add(1.0, q_norm.shape[0], n_doc_tokens_rand, len(doc_idxs))

    query_row = {
        'query_id': q_idx, 'doc_name': item['doc_name'],
        'domain': domain,  'question': question,
        'n_random_seeds': N_RANDOM_SEEDS,
    }
    for k in TOPK_EVAL:
        query_row[f'trad_r@{k}']    = round(m_trad[f'r{k}'], 4)
        query_row[f'trad_ndcg@{k}'] = round(m_trad[f'n{k}'], 4)

    # ── Random pruning — all ratios, averaged over N_RANDOM_SEEDS ─────────
    ratio_scores, ratio_token_counts = random_prune_topk(
        q_norm      = q_norm,
        content_idx = trad_idx,
        doc_matrix  = doc_mat,
        doc_mask    = doc_msk,
        topk_ratios = TOPK_RATIOS,
        n_seeds     = N_RANDOM_SEEDS,
    )

    for r in TOPK_RATIOS:
        key    = f"rand_r{int(r*100)}"
        scores = ratio_scores[r].cpu().tolist()
        n_doc_tokens_rand = doc_msk.sum().item()
        rand_efficiency.add(r, ratio_token_counts[r], n_doc_tokens_rand, len(doc_idxs))

        m = hit_metrics(top_k_indices(scores, max(TOPK_EVAL)), gt_local, TOPK_EVAL)
        record(rand_metrics, rand_domain_metrics, key, m, domain)
        for kk in TOPK_EVAL:
            query_row[f'{key}_r@{kk}']    = round(m[f'r{kk}'], 4)
            query_row[f'{key}_ndcg@{kk}'] = round(m[f'n{kk}'], 4)

    rand_query_rows.append(query_row)

print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND,
              title="Random Token Pruning Results")
rand_efficiency.report()

pd.DataFrame(rand_query_rows).to_csv(
    os.path.join(WORKING_DIR, "random_pruning_queries.csv"), index=False)
print("\n✅ Saved: random_pruning_queries.csv")

>>> METHOD 6: Random Token Pruning (Ablation Baseline)


Random Pruning:   0%|          | 0/1658 [00:00<?, ?it/s]


Random Token Pruning Results
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608
rand_r10                             48.36%  70.42%  78.07%  85.61%   0.5398   0.6356   0.6672   0.6944
rand_r20                             51.97%  74.38%  81.26%  88.13%   0.5784   0.6754   0.7044   0.7299
rand_r30                             53.53%  75.76%  82.58%  89.34%   0.5959   0.6907   0.7195   0.7442
rand_r40                             53.78%  76.37%  82.77%  89.15%   0.5995   0.6955   0.7224   0.7458
rand_r50                             54.70%  76.26%  83.38%  89.71%   0.6074   0.6990   0.7290   0.7521
rand_r60                             54.75%  76.69%  83.01%  89.26%   0.6080   0.7017   0.7284   0.7516
rand_r70                          

## Cell 12 — Final Summary & Save

In [18]:
# ==============================================================================
# FINAL SUMMARY — aggregate results from all methods
# ==============================================================================

print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

# ---- Traditional ----
print_summary(trad_metrics, trad_domain_metrics, ['traditional'],
              title="Traditional MaxSim")

# ---- Hierarchical best (show all ratios) ----
print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER,
              title="Hierarchical Ward Pooling")

# ---- Ours (show traditional and trad_weighted) ----
print_summary(ours_metrics, ours_domain_metrics, ['traditional', 'trad_weighted'],
              title="Ours — Baseline rows (add importance pkl for full ablation)")

# ---- Attention (L=1, trad scoring) ----
print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1)")

# ---- Spherical KMeans ----
print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS,
              title="Spherical KMeans Pooling")

# ---- Random Pruning ----
print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND,
              title="Random Token Pruning (Ablation Baseline)")

# ==============================================================================
# Save master summary CSVs
# ==============================================================================

def save_summary_csv(metrics, domain_metrics, method_keys, prefix):
    # Overall
    rows = []
    for key in method_keys:
        if key not in metrics: continue
        m = metrics[key]; cnt = m['count'] or 1
        row = {'method': key, 'count': cnt}
        for k in TOPK_EVAL:
            row[f'r@{k}']    = round(m[f'r{k}'] / cnt * 100, 4)
            row[f'ndcg@{k}'] = round(m[f'n{k}'] / cnt,       6)
        rows.append(row)
    df_sum = pd.DataFrame(rows)
    df_sum.to_csv(os.path.join(WORKING_DIR, f"{prefix}_summary.csv"), index=False)

    # Per-domain
    dom_rows = []
    for domain in sorted(domain_metrics):
        dm  = domain_metrics[domain]
        row = {'domain': domain}
        for key in method_keys:
            m_   = dm.get(key, _init_metric()); cnt_ = m_['count'] or 1
            for k in TOPK_EVAL:
                row[f'{key}_r{k}']    = round(m_[f'r{k}'] / cnt_ * 100, 4)
                row[f'{key}_ndcg{k}'] = round(m_[f'n{k}'] / cnt_,       6)
        dom_rows.append(row)
    pd.DataFrame(dom_rows).to_csv(
        os.path.join(WORKING_DIR, f"{prefix}_domain.csv"), index=False)

    print(f"✅ Saved: {prefix}_summary.csv and {prefix}_domain.csv")

save_summary_csv(trad_metrics, trad_domain_metrics, ['traditional'], "traditional")
save_summary_csv(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER, "hierarchical")
save_summary_csv(ours_metrics, ours_domain_metrics, ABLATION_KEYS_OURS, "ours_ablation")
save_summary_csv(attn_metrics, attn_domain_metrics, METHOD_KEYS_ATTN, "attention_pruning")
save_summary_csv(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS, "spherical_kmeans")
save_summary_csv(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND, "random_pruning")

print("\n>>> All evaluations complete.")

FINAL SUMMARY

Traditional MaxSim
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608

Hierarchical Ward Pooling
Method                                  R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
---------------------------------------------------------------------------------------------------
traditional                          56.09%  77.09%  83.47%  89.93%   0.6236   0.7104   0.7371   0.7608
hier_r10                             45.53%  68.84%  76.45%  86.15%   0.5066   0.6119   0.6447   0.6791
hier_r20                             52.82%  75.20%  82.69%  89.54%   0.5881   0.6819   0.7137   0.7389
hier_r30                             55.35%  77.11%  83.84%  90.85%   0.6176   0.7065   0.7345   0.7603
hie

## Cell 13 — Efficiency Analysis: FLOPs Comparison + Throughput Benchmark


In [19]:
# ==============================================================================
# EFFICIENCY ANALYSIS — Paper-grade metrics for all methods
#
# 1. FLOPs Comparison Table: deterministic, reproducible, noise-free.
#    Reports avg query token count (N_q), theoretical GFLOPs, FLOPs%
#    relative to traditional (full-token) baseline, and speedup factor.
#
# 2. Throughput Benchmark: empirical queries/sec under GPU-saturating
#    batched conditions.  Only runs for ratios that were collected.
# ==============================================================================

print("=" * 90)
print("EFFICIENCY ANALYSIS")
print("=" * 90)

# ── 1. FLOPs Comparison ──────────────────────────────────────────────────────

# Collect all trackers
all_eff_trackers = [
    ('Traditional MaxSim',     trad_efficiency),
    ('Ours (SVD+ClusterPool)', ours_efficiency),
    ('Hierarchical Ward',      hier_efficiency),
    ('Attention Pruning',      attn_efficiency),
    ('Spherical KMeans',       kmeans_efficiency),
    ('Random Pruning',         rand_efficiency),
]

# Print individual reports
for name, tracker in all_eff_trackers:
    tracker.report()

# ── Unified FLOPs comparison table ───────────────────────────────────────────

all_eff_dfs = []
for name, tracker in all_eff_trackers:
    df = tracker.to_dataframe()
    all_eff_dfs.append(df)

df_all_eff = pd.concat(all_eff_dfs, ignore_index=True)
df_all_eff.to_csv(os.path.join(WORKING_DIR, "efficiency_flops_all.csv"), index=False)
print(f"\n✅ Saved: efficiency_flops_all.csv")

# ── Cross-method comparison at each ratio (avg_Nq and speedup) ──────────────

print(f"\n{'='*90}")
print("Cross-Method Efficiency Comparison (avg query tokens at each ratio)")
print(f"{'='*90}")

# Build pivot: ratio × method → avg_Nq
pivot_rows = []
for name, tracker in all_eff_trackers:
    for ratio in sorted(tracker.data.keys()):
        entries = tracker.data[ratio]
        avg_nq  = np.mean([d['n_q'] for d in entries])
        pivot_rows.append({'ratio': ratio, 'method': name, 'avg_Nq': round(avg_nq, 1)})

df_pivot = pd.DataFrame(pivot_rows)
if not df_pivot.empty:
    pivot = df_pivot.pivot_table(index='ratio', columns='method', values='avg_Nq')
    print(pivot.to_string(float_format='%.1f'))
    pivot.to_csv(os.path.join(WORKING_DIR, "efficiency_nq_pivot.csv"))
    print(f"\n✅ Saved: efficiency_nq_pivot.csv")

# ── 2. Throughput Benchmark ───────────────────────────────────────────────────

print(f"\n{'='*90}")
print("Running Throughput Benchmark (batched MaxSim, GPU-saturating)...")
print(f"{'='*90}")

# Only run if we collected data
if throughput_bench.query_pool:
    tp_df = throughput_bench.run(
        doc_matrix, doc_mask,
        n_warmup=10, n_reps=50, batch_size=32
    )
    throughput_bench.report(tp_df)
    tp_df.to_csv(os.path.join(WORKING_DIR, "throughput_benchmark.csv"), index=False)
    print(f"\n✅ Saved: throughput_benchmark.csv")
else:
    print("No throughput data collected. Skipping benchmark.")

print("\n>>> Efficiency analysis complete.")

EFFICIENCY ANALYSIS

Efficiency Report — Traditional MaxSim
  Ratio        n   avg_Nq   std_Nq   avg_GFLOPs    FLOPs_%   Speedup
  ----------------------------------------------------------------------
  100%      1658     41.8      9.5       0.7328     100.0%     1.00×

Efficiency Report — Ours (SVD+ClusterPool)
  Ratio        n   avg_Nq   std_Nq   avg_GFLOPs    FLOPs_%   Speedup
  ----------------------------------------------------------------------
  10%        100      7.9      2.2       0.0713      17.6%     5.68×
  20%        100     15.5      4.4       0.1402      34.6%     2.89×
  30%        100     21.6      6.0       0.1951      48.2%     2.08×
  40%        100     26.9      7.4       0.2438      60.2%     1.66×
  50%        100     32.0      8.5       0.2892      71.4%     1.40×
  60%        100     36.1      9.6       0.3269      80.7%     1.24×
  70%        100     39.5     10.2       0.3577      88.3%     1.13×
  80%        100     42.4     11.0       0.3844      94.9%  